# VUR BERT Retrain -- reproducible pipeline

In [ ]:
import os, re, json, random, logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from transformers import (BertForSequenceClassification, AutoTokenizer,
                           Trainer, TrainingArguments)

## Config

In [ ]:
SEED = 42
INPUT_XLSX = 'verified_combined_vcug.xlsx'  # adjust to your local path
BASE_CHECKPOINT = 'dmis-lab/biobert-base-cased-v1.2'
OUTPUT_DIR = Path('YOUR_FOLDER_HERE')
VERSION = 'v2_split_before_downsample'

MAX_LENGTH = 512
UNDERSAMPLE_FRACTION = 0.25  # majority class ('No reflux') cut to this fraction, TRAIN ONLY

# matches UTD script's effective ratios: 80/20 then 80/20 again -> 64/16/20
TEST_SIZE = 0.20
VAL_SIZE_OF_REMAINDER = 0.20

TRAINING_HPARAMS = dict(
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=5e-5,       # now explicit -- matches the old implicit default
    optim='adamw_torch',      # now explicit -- matches the old implicit default
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='epoch',
)

SEVERITIES = ['0.0', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']

In [ ]:
# Defensive setup -- creates all needed folders up front, in case a library's
# auto-creation behavior differs across versions. Safe to re-run (idempotent).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / f'results_{VERSION}').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / f'logs_{VERSION}').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / f'model_{VERSION}').mkdir(parents=True, exist_ok=True)
print(f'All output folders ready under {OUTPUT_DIR.resolve()}')

## Seeding and logging

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def setup_logging(output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / f'train_log_{datetime.now():%Y%m%d_%H%M%S}.log'
    logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                         handlers=[logging.FileHandler(log_path), logging.StreamHandler()])
    logging.info(f'Logging to {log_path}')
    return log_path

set_all_seeds(SEED)
log_path = setup_logging(OUTPUT_DIR)
logging.info(f'Config: seed={SEED}, version={VERSION}')
logging.info(json.dumps(TRAINING_HPARAMS, indent=2))

## Text extraction (unchanged from the original notebook) and label filtering

In [ ]:
def get_reflux_sentences(df):
    impression_pattern = re.compile(
        r'FINDINGS:|Findings:|findings:|FINDING:|Finding:|finding:|findings/impression:|FINDINGS')
    df['text'] = df['Report Text'].apply(
        lambda x: impression_pattern.split(str(x))[1] if len(impression_pattern.split(str(x))) > 1 else str(x))
    df['text'] = df['text'].apply(lambda x: x.split('*****')[0] if len(x.split('*****')) > 1 else x)
    df['text'] = df['text'].apply(lambda x: x.split('END OF IMPRESSION')[0] if len(x.split('END OF IMPRESSION')) > 1 else x)
    df['text'] = df['text'].apply(
        lambda x: ' '.join([a for a in x.split(chr(10))
                             if 'reflux' in a.lower() or 'grade' in a.lower()
                             or 'vesicoureteral' in a.lower() or 'vur' in a.lower()]))
    df['text'] = df['text'].str.replace(r'\t|\n', ' ', regex=True)
    return df['text']

def filter_to_labeled_rows(df):
    before = len(df)
    df = df.dropna(subset=['Left', 'Right']).copy()
    keywords = ['Ungradeable', 'Duplicate?']
    pattern = '|'.join(re.escape(k) for k in keywords)
    df = df[~df['Left'].astype(str).str.contains(pattern, case=False, na=False)]
    df = df[~df['Right'].astype(str).str.contains(pattern, case=False, na=False)]
    logging.info(f'Filtered {before} -> {len(df)} labeled, gradeable rows.')
    return df.reset_index(drop=True)

def build_one_hot_label(row):
    vec = [0] * 20
    vec[SEVERITIES.index(str(float(row['Left'])))] = 1
    vec[10 + SEVERITIES.index(str(float(row['Right'])))] = 1
    return vec

df = pd.read_excel(INPUT_XLSX)
df = filter_to_labeled_rows(df)
df['text'] = get_reflux_sentences(df)
df['one_hot'] = df.apply(build_one_hot_label, axis=1)
df.head()

## Split FIRST (on the full, non-downsampled data)

In [ ]:
def make_split(df, seed):
    strat_key = df['Left'].astype(str).values
    train_val_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=seed, stratify=strat_key)
    strat_key_2 = train_val_df['Left'].astype(str).values
    train_df, val_df = train_test_split(train_val_df, test_size=VAL_SIZE_OF_REMAINDER, random_state=seed, stratify=strat_key_2)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, val_df, test_df = make_split(df, seed=SEED)
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

### Split diagnostics -- full severity distribution per split (Left side shown; Right is analogous)

Worth checking this table before training: since the split is stratified on the Left-side grade only, this confirms train/val/test all reflect the same underlying severity distribution as the full dataset -- i.e. the split didn't accidentally concentrate rare grades into one partition.

In [ ]:
def split_distribution_table(splits):
    rows = []
    for sev in SEVERITIES:
        row = {'severity': sev}
        for name, split_df in splits.items():
            frac = (split_df['Left'].astype(str) == sev).mean()
            row[f'{name} (n={len(split_df)})'] = f'{frac:.3f}'
        rows.append(row)
    return pd.DataFrame(rows)

splits = {'full': df, 'train': train_df, 'val': val_df, 'test': test_df}
split_distribution_table(splits)

## Downsample -- ONLY the train split, explicitly seeded

In [ ]:
def downsample_train_only(train_df, seed):
    labels_str = train_df['Left'].astype(str) + '_' + train_df['Right'].astype(str)
    counts = labels_str.value_counts().to_dict()
    majority_class = max(counts, key=counts.get)
    target_counts = dict(counts)
    target_counts[majority_class] = int(counts[majority_class] * UNDERSAMPLE_FRACTION)
    logging.info(f'Downsampling train only. Majority class: {majority_class} '
                 f'({counts[majority_class]} -> {target_counts[majority_class]})')
    undersampler = RandomUnderSampler(sampling_strategy=target_counts, random_state=seed)
    idx_2d = np.arange(len(train_df)).reshape(-1, 1)
    idx_resampled, _ = undersampler.fit_resample(idx_2d, labels_str.values)
    return train_df.iloc[idx_resampled.ravel()].reset_index(drop=True)

train_df_downsampled = downsample_train_only(train_df, seed=SEED)
print(f'train size after downsampling: {len(train_df_downsampled)} (was {len(train_df)})')

## Save train/val/test splits to disk

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, split_df in [('train', train_df_downsampled), ('val', val_df), ('test', test_df)]:
    out_path = OUTPUT_DIR / f'split_{name}_{VERSION}.csv'
    split_df[['Accession Number', 'text', 'Left', 'Right']].to_csv(out_path, index=False)
    print(f'Saved {name} split ({len(split_df)} rows) to {out_path}')

In [ ]:
# --- SMOKE TEST TOGGLE -- set to False before moving to the real server ---
SMOKE_TEST = True

if SMOKE_TEST:
    train_df_downsampled = train_df_downsampled.sample(n=min(50, len(train_df_downsampled)), random_state=SEED).reset_index(drop=True)
    val_df_smoke = val_df.sample(n=min(20, len(val_df)), random_state=SEED).reset_index(drop=True)
    test_df_smoke = test_df.sample(n=min(20, len(test_df)), random_state=SEED).reset_index(drop=True)
    val_df, test_df = val_df_smoke, test_df_smoke  # only overwrite in-memory copies, saved CSVs are untouched

    TRAINING_HPARAMS['num_train_epochs'] = 1
    print(f"SMOKE TEST active -- train={len(train_df_downsampled)}, val={len(val_df)}, test={len(test_df)}, epochs=1")

## Dataset / Trainer classes

In [ ]:
class VURDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(list(texts), truncation=True, padding='max_length',
                                    max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.float)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

class MultiTaskClassificationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs[0]
        loss = torch.nn.functional.cross_entropy(logits, labels)
        return (loss, outputs) if return_outputs else loss

## Train

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_CHECKPOINT, model_max_length=MAX_LENGTH)
model = BertForSequenceClassification.from_pretrained(BASE_CHECKPOINT, num_labels=20)

train_dataset = VURDataset(train_df_downsampled['text'], list(train_df_downsampled['one_hot']), tokenizer)
val_dataset = VURDataset(val_df['text'], list(val_df['one_hot']), tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / f'results_{VERSION}'),
    seed=SEED,
    **TRAINING_HPARAMS,
)

trainer = MultiTaskClassificationTrainer(model=model, args=training_args,
                                          train_dataset=train_dataset, eval_dataset=val_dataset)

train_result = trainer.train()
logging.info(f'Training complete. Final metrics: {train_result.metrics}')

## Save model + config

In [ ]:
final_model_dir = OUTPUT_DIR / f'model_{VERSION}'
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

with open(final_model_dir / 'training_config.json', 'w') as f:
    json.dump({'seed': SEED, 'base_checkpoint': BASE_CHECKPOINT, 'training_hparams': TRAINING_HPARAMS,
               'test_size': TEST_SIZE, 'val_size_of_remainder': VAL_SIZE_OF_REMAINDER,
               'undersample_fraction': UNDERSAMPLE_FRACTION, 'train_n': len(train_df_downsampled),
               'val_n': len(val_df), 'test_n': len(test_df)}, f, indent=2)
print(f'Saved model + config to {final_model_dir}')

## Run inference on the held-out (never-downsampled, never-trained-on) test set

In [ ]:
def predict_one(text, model, tokenizer, device):
    tokens = tokenizer(str(text), add_special_tokens=True, max_length=MAX_LENGTH,
                        truncation=True, padding='max_length', return_tensors='pt')
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        logits = model(**tokens).logits.squeeze(0).cpu().numpy()
    left_idx = int(logits[:10].argmax())
    right_idx = int(logits[10:].argmax())
    return float(SEVERITIES[left_idx]), float(SEVERITIES[right_idx])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

left_preds, right_preds = [], []
for i, row in test_df.iterrows():
    l, r = predict_one(row['text'], model, tokenizer, device)
    left_preds.append(l)
    right_preds.append(r)

test_df['BERT_Left'] = left_preds
test_df['BERT_Right'] = right_preds
test_df.to_csv(OUTPUT_DIR / f'test_predictions_{VERSION}.csv', index=False)
print('Saved test predictions.')

## Metrics: Wilson per-label, macro/weighted + patient-level bootstrap, ordinal (QWK/MAE)

Self-contained -- no external imports beyond sklearn/numpy/pandas, so this section runs the same way in either notebook.

In [ ]:
import math
import random
import numpy as np
from sklearn.metrics import cohen_kappa_score

SEVERITIES = ['0.0', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']
SEVERITY_RANK = {v: i for i, v in enumerate([float(s) for s in SEVERITIES])}
VMETRICS = ['precision', 'recall', 'f1', 'accuracy']


def wilson_ci(x, n, z=1.96):
    """95% Wilson score interval for a binomial proportion x/n."""
    if n <= 0:
        return (0.0, 0.0)
    phat = x / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (z * math.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def per_class_metrics(tp, fp, fn, tn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    return precision, recall, f1, accuracy

In [ ]:
def build_label_confusion(df, lcol, rcol, true_lcol='Left', true_rcol='Right'):
    """20 one-vs-rest confusion matrices (10 severities x 2 sides) from patient rows."""
    d = {}
    for sev in SEVERITIES:
        sev_f = float(sev)
        for side, true_col, pred_col in [('Left', true_lcol, lcol), ('Right', true_rcol, rcol)]:
            true_vals = df[true_col].astype(float).values
            pred_vals = df[pred_col].astype(float).values
            tp = fp = fn = tn = 0
            for t, p in zip(true_vals, pred_vals):
                if t == sev_f and p == sev_f:
                    tp += 1
                elif t != sev_f and p == sev_f:
                    fp += 1
                elif t == sev_f and p != sev_f:
                    fn += 1
                else:
                    tn += 1
            d[f'{sev}_{side}'] = dict(tp=tp, fp=fp, fn=fn, tn=tn)
    return d


def per_label_all_metrics(d_raw):
    out = {}
    for label, counts in d_raw.items():
        tp, fp, fn, tn = counts['tp'], counts['fp'], counts['fn'], counts['tn']
        p, r, f1, a = per_class_metrics(tp, fp, fn, tn)
        out[label] = dict(precision=(p, tp, tp + fp), recall=(r, tp, tp + fn),
                           f1=(f1, 2 * tp, 2 * tp + fp + fn), accuracy=(a, tp + tn, tp + tn + fp + fn))
    return out


def vcug_aggregate_across_labels(d_raw, labels):
    per_class = {}
    for label in labels:
        c = d_raw[label]
        tp, fp, fn, tn = c['tp'], c['fp'], c['fn'], c['tn']
        p, r, f1, a = per_class_metrics(tp, fp, fn, tn)
        per_class[label] = dict(precision=p, recall=r, f1=f1, accuracy=a, support=tp + fn)
    macro, weighted = {}, {}
    total = sum(per_class[l]['support'] for l in labels)
    for key in ['precision', 'recall', 'accuracy']:
        macro[key] = sum(per_class[l][key] for l in labels) / len(labels)
        weighted[key] = (sum(per_class[l][key] * per_class[l]['support'] for l in labels) / total) if total > 0 else 0
    macro['f1'] = (2 * macro['precision'] * macro['recall']) / (macro['precision'] + macro['recall']) if (macro['precision'] + macro['recall']) > 0 else 0
    weighted['f1'] = (2 * weighted['precision'] * weighted['recall']) / (weighted['precision'] + weighted['recall']) if (weighted['precision'] + weighted['recall']) > 0 else 0
    return macro, weighted


def patient_bootstrap_ci(df, lcol, rcol, true_lcol='Left', true_rcol='Right', n_boot=2000, seed=0):
    rng = random.Random(seed)
    n = len(df)
    boot = {'macro': {m: [] for m in VMETRICS}, 'weighted': {m: [] for m in VMETRICS}}
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        sample = df.iloc[idx].reset_index(drop=True)
        d_boot = build_label_confusion(sample, lcol, rcol, true_lcol, true_rcol)
        macro, weighted = vcug_aggregate_across_labels(d_boot, list(d_boot.keys()))
        for m in VMETRICS:
            boot['macro'][m].append(macro[m])
            boot['weighted'][m].append(weighted[m])
    def pct(vals):
        vals = sorted(vals)
        lo = vals[int(0.025 * len(vals))]
        hi = vals[min(len(vals) - 1, int(0.975 * len(vals)))]
        return lo, hi
    return {conv: {m: pct(boot[conv][m]) for m in VMETRICS} for conv in ['macro', 'weighted']}


def ordinal_metrics(true_vals, pred_vals):
    """Quadratic-weighted kappa + mean absolute error in grade units."""
    true_ranks = [SEVERITY_RANK[float(v)] for v in true_vals]
    pred_ranks = [SEVERITY_RANK[float(v)] for v in pred_vals]
    qwk = cohen_kappa_score(true_ranks, pred_ranks, weights='quadratic')
    mae = np.mean(np.abs(np.array(true_vals, dtype=float) - np.array(pred_vals, dtype=float)))
    return qwk, mae


def print_full_metrics(df, lcol, rcol, model_name, true_lcol='Left', true_rcol='Right', n_boot=2000):
    """Runs the complete metric suite and prints a manuscript-style summary."""
    print(f'{"="*70}\n{model_name}\n{"="*70}')

    d_raw = build_label_confusion(df, lcol, rcol, true_lcol, true_rcol)
    per_label = per_label_all_metrics(d_raw)

    print('\n-- Wilson per-label (precision) --')
    for label, m in per_label.items():
        val, x, n = m['precision']
        lo, hi = wilson_ci(x, n)
        print(f'  {label:12s} precision = {val:.4f} [{lo:.4f}, {hi:.4f}]  (n={n})')

    macro, weighted = vcug_aggregate_across_labels(d_raw, list(d_raw.keys()))
    boot_ci = patient_bootstrap_ci(df, lcol, rcol, true_lcol, true_rcol, n_boot=n_boot)

    print('\n-- Macro / weighted aggregate (patient-level bootstrap) --')
    for conv, point in [('macro', macro), ('weighted', weighted)]:
        for metric in VMETRICS:
            lo, hi = boot_ci[conv][metric]
            print(f'  {conv:8s} {metric:10s} = {point[metric]:.4f} [{lo:.4f}, {hi:.4f}]')

    print('\n-- Ordinal metrics (QWK, MAE in grade units) --')
    for side, true_col, pred_col in [('Left', true_lcol, lcol), ('Right', true_rcol, rcol)]:
        qwk, mae = ordinal_metrics(df[true_col].values, df[pred_col].values)
        print(f'  {side:6s} QWK = {qwk:.4f}   MAE = {mae:.4f}')
    true_both = list(df[true_lcol].values) + list(df[true_rcol].values)
    pred_both = list(df[lcol].values) + list(df[rcol].values)
    qwk_c, mae_c = ordinal_metrics(true_both, pred_both)
    print(f'  {"Both":6s} QWK = {qwk_c:.4f}   MAE = {mae_c:.4f}')

    return {'per_label': per_label, 'macro': macro, 'weighted': weighted,
            'bootstrap_ci': boot_ci, 'd_raw': d_raw}

## Run the full metric suite on the test set

In [ ]:
results = print_full_metrics(test_df, 'BERT_Left', 'BERT_Right', 'BERT (retrained, held-out test)',
                              true_lcol='Left', true_rcol='Right', n_boot=2000)